# LP Formulation for Detecting Argmaxable Classes

We consider a linear classifier with weight matrix  

$$W \in \mathbb{R}^{n \times d}$$

where each row $w_c \in \mathbb{R}^d$ corresponds to class $c$.

Prediction rule:

$$\hat{c}(z) = \arg\max_{c} \; w_c^\top z$$

Our goal is:

> Determine whether a given class $c^*$ can be a **strict argmax** for some bounded feature vector $z$.

---

## Step 1 — Margin Condition

Class $c^*$ strictly wins at $z$ if

$$w_{c^*}^\top z \ge w_c^\top z + t \quad \forall c \neq c^*$$

where $t > 0$ is a **margin**.

Rewriting:

$$(w_{c^*} - w_c)^\top z \ge t \quad \forall c \neq c^*$$

This means:
* Class $c^*$ beats every competitor
* By at least margin $t$

---

## Step 2 — Optimization Problem

We search for the largest achievable margin inside a bounded box:

$$\max_{z,t} \quad t$$

subject to:

$$(w_{c^*} - w_c)^\top z \ge t \quad \forall c \neq c^*$$

$$-z_{\text{box}} \le z_i \le z_{\text{box}} \quad \forall i$$

---

## Step 3 — Convert to Standard LP Form

Since `linprog` solves minimization problems with

$$A_{ub} x \le b_{ub}$$

we rewrite the constraints:

$$(w_c - w_{c^*})^\top z + t \le 0$$

Decision variable:

$$x = \begin{bmatrix} z \\ t \end{bmatrix} \in \mathbb{R}^{d+1}$$

Objective (maximize $t$) becomes:

$$\min (-t)$$

---

## Final Linear Program

$$\min_{z,t} \quad -t$$

subject to:

$$(w_c - w_{c^*})^\top z + t \le 0 \quad \forall c \neq c^*$$

$$-z_{\text{box}} \le z_i \le z_{\text{box}}$$

---

## Interpretation of the Solution

Let $t^*$ be the optimal value.

* If $t^* > 0$:  
  Class $c^*$ is **argmaxable** —  
  there exists some bounded $z$ where it strictly wins.

* If $t^* \le 0$:  
  Class $c^*$ can never be a unique argmax within the box.

---

## Geometric Meaning

Each constraint defines a half-space:

$$(w_{c^*} - w_c)^\top z \ge t$$

We are searching for a point $z$ in the box where all these half-spaces overlap with positive margin.

Thus the LP computes:

$$t^* = \max_{z \in [-z_{\text{box}},z_{\text{box}}]^d} \min_{c\neq c^*} (w_{c^*}-w_c)^\top z.$$

---

## Role of `z_box`

Without bounding $z$, the margin could grow arbitrarily large by scaling $z$.

The box constraint ensures:
* The problem is bounded
* We measure margin within a controlled feature region

Larger `z_box` ⇒ potentially larger margins.

---

## Practical Insight

The experiment shows a clear phase behavior:

* When $n/d$ is large (many classes, low dimension),  
  few classes are argmaxable.
* When $d$ increases relative to $n$,  
  almost all classes become argmaxable.

In [6]:
import numpy as np
from scipy.optimize import linprog
from time import time


def margin_lp_for_class(W, cstar, z_box=1.0):

    n, d = W.shape
    w_star = W[cstar]

    # c = (0,0,0,...,0,-1)
    c_obj = np.zeros(d + 1)
    c_obj[-1] = -1.0

    A = []
    b = []

    # skip the target class itself
    for c in range(n):
        if c == cstar:
            continue

        # for the constraint Ax <= b
        # A = [w_c - w_star, 1], x = [z, t]^T, b = 0
        row = np.zeros(d + 1)
        row[:d] = W[c] - w_star
        row[-1] = 1.0
        A.append(row)
        b.append(0.0)

    A = np.array(A)
    b = np.array(b)

    # bounds = [(-z_box, z_box), (-z_box, z_box), ..., (None, None)] since x = [z_1, z_2, ..., t]
    bounds = [(-z_box, z_box)] * d + [(None, None)]

    res = linprog(c=c_obj, A_ub=A, b_ub=b, bounds=bounds, method="highs")

    if not res.success:
        return None

    return res.x[-1]


def argmaxable_classes(W, tol=1e-6):
    n, _ = W.shape
    is_arg = np.zeros(n, dtype=bool)

    for cstar in range(n):
        t_star = margin_lp_for_class(W, cstar)

        if t_star is not None and t_star > tol:
            is_arg[cstar] = True

    return is_arg


def run_experiment(n_list, d_list, trials=5, seed=42):
    rng = np.random.default_rng(seed)

    for n in n_list:
        for d in d_list:

            fractions = []
            start = time()

            for _ in range(trials):
                W = rng.standard_normal((n, d))
                is_arg = argmaxable_classes(W)
                fractions.append(is_arg.mean())

            elapsed = time() - start

            print("=" * 60)
            print(f"n = {n}, d = {d}, n/d = {n/d:.2f}")
            print(f"Mean argmaxable fraction: {np.mean(fractions):.4f}")
            print(f"Std over trials: {np.std(fractions):.4f}")
            print(f"Time: {elapsed:.2f}s")


if __name__ == "__main__":

    n_list = [10, 20, 40, 80]
    d_list = [2, 4, 8, 16]

    run_experiment(n_list, d_list, trials=3)

n = 10, d = 2, n/d = 5.00
Mean argmaxable fraction: 0.6333
Std over trials: 0.0943
Time: 0.04s
n = 10, d = 4, n/d = 2.50
Mean argmaxable fraction: 1.0000
Std over trials: 0.0000
Time: 0.01s
n = 10, d = 8, n/d = 1.25
Mean argmaxable fraction: 1.0000
Std over trials: 0.0000
Time: 0.01s
n = 10, d = 16, n/d = 0.62
Mean argmaxable fraction: 1.0000
Std over trials: 0.0000
Time: 0.01s
n = 20, d = 2, n/d = 10.00
Mean argmaxable fraction: 0.3000
Std over trials: 0.0408
Time: 0.02s
n = 20, d = 4, n/d = 5.00
Mean argmaxable fraction: 0.8333
Std over trials: 0.0850
Time: 0.02s
n = 20, d = 8, n/d = 2.50
Mean argmaxable fraction: 1.0000
Std over trials: 0.0000
Time: 0.02s
n = 20, d = 16, n/d = 1.25
Mean argmaxable fraction: 1.0000
Std over trials: 0.0000
Time: 0.02s
n = 40, d = 2, n/d = 20.00
Mean argmaxable fraction: 0.1667
Std over trials: 0.0312
Time: 0.04s
n = 40, d = 4, n/d = 10.00
Mean argmaxable fraction: 0.6167
Std over trials: 0.0624
Time: 0.04s
n = 40, d = 8, n/d = 5.00
Mean argmaxable fra